# KAT-VAD v2 — **DoTA-CAP**: DoTA pixels from CAP-DATA → VideoMAE (no training)

Addendum §11 (**Amendment 4**, D13–D15, L1–L7 — fixed before any motion score). MM-AU Phase 0 found **1,248 / 1,397**
DoTA clips inside CAP-DATA at κ ≥ 0.99. This notebook turns them into the motion endpoint:

| step | what | where |
|---|---|---|
| 0 | setup, deps | CPU |
| 1 | `dota_cap align` — per-frame CAP map per DoTA clip from the two s1 CLIP caches; gates L1–L3 | CPU, ~minutes |
| 2 | `dota_cap extract` per CAP group — stream **only** the needed CAP videos from the Drive parts, rebuild DoTA's frame sequence, **pixel gate** (CLIP of the rebuilt frames == `DoTA_s1_ncc`, L4–L5), then VideoMAE V2-B and V2-S at DoTA's native 10 fps (L6) | **GPU (L4)** |
| 3 | `dota_cap finalize` — the DoTA-CAP id list + coverage (L7) | CPU |

No label and no score is read. DoTA-eval ids are never read (coverage prints `rest_eval` = all − dev).
Every step resumes: `align` is cheap, `extract` keeps a per-group report and skips decided clips.
Output: `Thesis/cache/video/<encoder>/DoTA_CAP_s1_squash/{dota_id}.npy`, row-aligned with `DoTA_s1_ncc`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'
os.environ['PROJECT_ROOT']       = DRIVE_V2
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_V2}/outputs'

os.environ['REPO'] = f'{DRIVE_V2}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v2 there'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'
assert hasattr(constants, 'DOTA_CAP_DATASET'), 'this upload predates Amendment 4 -- re-upload branch v2'

ENCODERS = [constants.VIDEOMAE_ENCODER_B, constants.VIDEOMAE_ENCODER_S]   # one pass over the tar for both (E2(d))
GROUPS   = ['1-10', '11', '12-42', '43', '44-62']
BATCH_GB = constants.MMAU_STREAM_BATCH_GB

RAW      = constants.DATA_ROOT / 'MMAU' / 'raw'
CAP_CLIP = constants.CLIP_CACHE_DIR / f'{constants.MMAU_CAP_DATASET}_s1_ncc'
DOTA_S1  = constants.CLIP_CACHE_DIR / 'DoTA_s1_ncc'
MATCHES  = constants.OUTPUT_ROOT / 'REPORTS' / 'mmau_p0' / 'all' / 'mmau_p0_matches.json'
REPORT   = constants.OUTPUT_ROOT / 'REPORTS' / 'dota_cap'
LOCAL    = Path('/content/dcap')
for p in (REPORT, LOCAL):
    p.mkdir(parents=True, exist_ok=True)
for name, path in (('CAP s1', CAP_CLIP), ('DoTA s1', DOTA_S1), ('P0 matches', MATCHES),
                   ('CAP annotation', RAW / 'cap_text_annotations.xls')):
    assert path.exists(), f'missing {name}: {path}'
    print(f'  {name:15s} {path}   OK')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu xlrd
python -c "import av, einops, faiss, xlrd; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- step 2 needs one'
df -h /content | tail -1

In [ ]:
import json
import shutil
import subprocess
import time


def stream(cmd, name):
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print(''.join(tail))
        raise RuntimeError(f'{name} exited {proc.returncode}')

## 1. Align (L1–L3) — CLIP caches only

Only the CAP clips that P0 matched **exact** and the 1,397 DoTA rows are needed; they are staged to VM disk first
(reading thousands of small `.npy` from Drive inside the tool is slow).

In [ ]:
t0 = time.time()
matches = json.loads(MATCHES.read_text())['dota']
need_cap = {m['cap_id'] for m in matches if m['grade'] == 'exact'}
stage = LOCAL / 'stage'
for src, name, ids in ((CAP_CLIP, 'cap', need_cap), (DOTA_S1, 'dota_s1', None)):
    dst = stage / name
    dst.mkdir(parents=True, exist_ok=True)
    for p in (src.glob('*.npy') if ids is None else (src / f'{i}.npy' for i in ids)):
        if not (dst / p.name).exists():
            shutil.copy2(p, dst / p.name)
print(f'staged {len(need_cap)} CAP + {len(list((stage / "dota_s1").glob("*.npy")))} DoTA in {time.time() - t0:.0f}s')

stream([sys.executable, '-m', 'core.tools.dota_cap', 'align', '--matches', MATCHES,
        '--dota-clip-dir', stage / 'dota_s1', '--cap-clip-dir', stage / 'cap', '--out-dir', REPORT],
       'dota_cap align')
print((REPORT / 'dota_cap_align.md').read_text())

## 2. Extract (L4–L6) per CAP group — GPU

For each group: the CAP ids of the aligned clips that live in it (from the annotation's `type`) go to `--cap-ids-file`,
so the tool writes only those videos and **stops reading the tar once the last one is closed**. Each DoTA clip is rebuilt
as a symlink folder, CLIP-checked against `DoTA_s1_ncc` (a clip that fails is reported, gets no feature file), then
encoded by every encoder in `ENCODERS`. Peak disk ≈ one batch (`BATCH_GB`) + one video.

In [ ]:
# DoTA_s1_ncc is read straight from Drive (one .npy per clip): a fresh GPU runtime needs only
# step 0 here -- step 1's staging lived on the CPU runtime's disk; the alignment is on Drive
import pandas as pd

ann = pd.read_excel(RAW / 'cap_text_annotations.xls', sheet_name='annotation file')
type_of = {f'{v:06d}': int(t) for v, t in zip(ann['video'].astype(int), ann['type'].astype(int))}
alignment = json.loads((REPORT / 'dota_cap_alignment.json').read_text())
kept_cap = {c['cap_id'] for c in alignment['clips'].values() if c['reason'] == 'ok'}
assert kept_cap <= type_of.keys(), sorted(kept_cap - type_of.keys())[:5]

free_gb = shutil.disk_usage('/content').free / 1e9
assert free_gb >= 3 * BATCH_GB, f'{free_gb:.0f} GB free on /content, need {3 * BATCH_GB:.0f} -- lower BATCH_GB'

for g in GROUPS:
    lo, hi = (int(x) for x in (g.split('-') * 2)[:2])
    ids = sorted(c for c in kept_cap if lo <= type_of[c] <= hi)
    print(f'group {g}: {len(ids)} CAP videos needed')
    if not ids:
        continue
    ids_file = LOCAL / f'cap_ids_{g}.txt'
    ids_file.write_text(''.join(f'{i}\n' for i in ids))
    parts = sorted((RAW / 'CAP-DATA_chunks' / g).glob(f'{g}.part_*'))
    assert parts, f'no parts for group {g}'
    t0 = time.time()
    stream([sys.executable, '-m', 'core.tools.dota_cap', 'extract',
            '--alignment', REPORT / 'dota_cap_alignment.json', '--parts', *parts,
            '--work-dir', LOCAL / 'work' / g, '--dota-clip-dir', DOTA_S1,
            '--report', REPORT / f'extract_{g}.json', '--cap-ids-file', ids_file,
            '--encoders', *ENCODERS, '--batch-size', '32', '--max-batch-gb', str(BATCH_GB),
            '--device', 'auto'], 'dota_cap extract')
    print(f'group {g}: {(time.time() - t0) / 60:.0f} min')

## 3. Finalize (L7) → the DoTA-CAP id list

In [ ]:
reports = sorted(REPORT.glob('extract_*.json'))
stream([sys.executable, '-m', 'core.tools.dota_cap', 'finalize',
        '--alignment', REPORT / 'dota_cap_alignment.json', '--reports', *reports,
        '--encoders', *ENCODERS, '--out-dir', REPORT], 'dota_cap finalize')
print((REPORT / 'dota_cap_readout.md').read_text())
for enc in ENCODERS:
    out = constants.VIDEO_CACHE_DIR / enc / f'{constants.DOTA_CAP_DATASET}_s1_squash'
    print(enc, len(list(out.glob('*.npy'))), 'feature files', out)

## 4. Record

Paste `dota_cap_align.md` and `dota_cap_readout.md` back into the chat, and bring `dota_cap_ids.txt`
(`Thesis-V2/outputs/REPORTS/dota_cap/`) to the repo: it is frozen into `core/splits/v2/` **before any motion score is
read** (Amendment 4, L7).